# A2 — Knowledge-Base Demo

Two authors, two sections, evidence for A2 form Section 5 ("Evidence It Works"):

- **Section 1 (below) — OCR quality.** Real Tesseract output scored against real gold labels
  in `grading_kit/labels.jsonl`, on the actual corpus. Not mocked, not a demo — this ran against
  real scanned pages.
- **Sections 2-6 — Stage 4 (chunk → embed → store — `index/chunk.py`, `index/embed.py`,
  `index/store.py`).** The demo corpus, the real index pipeline, index statistics, one retrieval
  example, and one retrieval-level worst failure.
- **Section 7 — Person B placeholder** for the one remaining item: an OCR-level worst-failure
  example (OCR quality itself is now done in Section 1 below).

**Scope note for Sections 2-6:** they run against a **small, hand-written demonstration corpus**,
not the full real corpus Section 1 uses — Stage 4's *own* real-corpus run
(`scripts/get_data.sh` → `scripts/build_index.sh`) is still blocked independently of OCR, on two
upstream stub fixes confirmed by `tests/test_retrieval.py`'s
`test_build_knowledge_base_blocked_by_enhance_stub_with_current_config` and
`test_build_knowledge_base_blocked_by_pii_stub_once_enhance_disabled`: `ingest/enhance.py` and
`governance/pii.py` both still raise `NotImplementedError` on the real
`pipeline.build_knowledge_base()` entry point, before Stage 4 is ever reached. Not
index/chunk/embed/store's files to fix — see `configs/design_choices.md`'s Stage 4 row and
`reports/pipeline_diagram.md` for the full picture. Sections 2-6 are still real, unmocked code
(the actual `multilingual-e5-base` model, a real FAISS index) — only the *input text* in Section 2
is synthetic, standing in for what Section 1's real OCR pipeline would have produced from more
scanned pages than the demo needs.

## OCR Quality — CER / WER against the gold standard
Evaluates Stage 3's actual OCR accuracy against `grading_kit/labels.jsonl` (the independently
human-reviewed page sample) — a continuous accuracy measurement, distinct from the pass/fail
CER gate in `vision/ocr.py` (which only tags a line `"gold"` on an exact CER==0.0 match). A
line that fails that strict gate is still scored here, so this reports the real OCR error
rate, not just the gate's pass rate.

CER = character-level edit distance / reference length. WER is the same idea at word
granularity: WER = word-level edit distance / reference word count.

In [1]:
import sys
from pathlib import Path

# Find the project root and add the 'src' directory to sys.path
current = Path.cwd().resolve()
while current != current.parent:
    src_dir = current / "src"
    if src_dir.exists() and (src_dir / "doc_agent").exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
    current = current.parent

import json
from doc_agent.ingest import loader, preprocess
from doc_agent.vision import layout, ocr
from doc_agent.vision.ocr import _normalize, _levenshtein

In [2]:
import json
import sys
from pathlib import Path

try:
    import yaml
except ImportError:
    !pip install pyyaml
    import yaml

# Find project root directory dynamically (searches upwards for configs or src)
current = Path.cwd().resolve()
project_root = current
while current != current.parent:
    if (current / "configs").exists() or (current / "src").exists():
        project_root = current
        break
    current = current.parent

# Load configuration relative to project root
config_path = project_root / "configs" / "config.yaml"

if config_path.exists():
    with open(config_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    print(f"Loaded configuration from: {config_path}")
else:
    cfg = {}
    print(f"Warning: Config file not found at {config_path}")

# Load gold standard labels relative to project root
labels_rel = cfg.get("grading_kit", {}).get("labels_path", "grading_kit/labels.jsonl")
labels_path = Path(labels_rel)
if not labels_path.is_absolute():
    labels_path = project_root / labels_path

gold_texts: dict[str, str] = {}
if labels_path.exists():
    with open(labels_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            row = json.loads(line)
            pid, text = row.get("page_id"), row.get("text", "")
            if pid and text and not text.startswith("REPLACE ME"):
                gold_texts[pid] = text

print(f"Loaded {len(gold_texts)} gold-labelled pages from {labels_path}")
if not gold_texts:
    print(
        "No gold labels yet -- fill grading_kit/labels.jsonl with reviewed page "
        "transcriptions before this section can report anything."
    )

Loaded configuration from: /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/configs/config.yaml
Loaded 5 gold-labelled pages from /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/grading_kit/labels.jsonl


In [3]:
# 1. Ensure configuration paths use resolved absolute paths
cfg.setdefault("paths", {})
cfg["paths"]["raw_dir"] = str((project_root / "data" / "raw").resolve())
cfg["paths"]["processed_dir"] = str(
    (project_root / "data" / "processed").resolve()
)

# 2. Guarantee the processed output directory exists
processed_dir = Path(cfg["paths"]["processed_dir"]).resolve()
processed_dir.mkdir(parents=True, exist_ok=True)

try:
    # 3. Run Stages 1-3 only for gold-labelled pages
    raw_pages = [p for p in loader.load_pages(cfg) if p.id in gold_texts]
    print(f"Raw pages: {len(raw_pages)}")
    pages = preprocess.run(raw_pages, cfg)
    regions = layout.detect(pages, cfg)
    ocr.transcribe(
        regions, cfg
    )  # writes data/processed/ocr_meta.jsonl + layout_meta.jsonl

    # 4. Parse OCR outputs safely using a context manager
    ocr_file = processed_dir / "ocr_meta.jsonl"
    ocr_rows = []
    if ocr_file.exists():
        with open(ocr_file, encoding="utf-8") as f:
            ocr_rows = [json.loads(line) for line in f if line.strip()]

    by_page: dict[str, list[str]] = {}
    for row in sorted(ocr_rows, key=lambda r: r["region_id"]):
        by_page.setdefault(row["page_id"], []).append(
            row["ocr_text_normalized"]
        )

    print(f"OCR'd {len(pages)} gold-labelled pages, {len(ocr_rows)} lines total")

except FileNotFoundError as e:
    by_page = {}
    print("Skipping -- corpus not available yet in this environment:")
    print(" ", e)

{"ts":"2026-08-10 21:47:54,017","lvl":"INFO","mod":"doc_agent.ingest.loader","msg":"loaded 833 pages across 6 documents from /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/data/raw"}
Raw pages: 5
{"ts":"2026-08-10 21:47:59,935","lvl":"INFO","mod":"doc_agent.ingest.preprocess","msg":"preprocessed 5 pages, dropped 0 blank/separator pages"}
{"ts":"2026-08-10 21:48:02,965","lvl":"INFO","mod":"doc_agent.vision.layout","msg":"detected 145 line regions across 5 pages -> /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/data/processed/layout_meta.jsonl"}
{"ts":"2026-08-10 21:48:11,111","lvl":"INFO","mod":"doc_agent.vision.ocr","msg":"OCR'd 145 regions across 5 pages: 0 accepted -> 0 chunks, 145 rejected -> /home/suchi/Downloads/DL/doc-agent-template/doc-agent-starter/data/processed/ocr_meta.jsonl"}
OCR'd 5 gold-labelled pages, 145 lines total


In [4]:
# 3. Score CER/WER: keep the line-level gate for layout validation, but also compute a page-level
# score so the notebook still reports meaningful OCR quality when the detected line count does not
# exactly match the gold page transcription.
def _wer(hyp_words: list[str], ref_words: list[str]) -> float:
    """Word Error Rate = word-level edit distance / reference word count."""
    if not ref_words:
        raise ValueError('_wer() requires a non-empty reference')
    return _levenshtein(hyp_words, ref_words) / len(ref_words)

line_results = []
page_results = []
mismatched_pages = []
for page_id, gold_text in gold_texts.items():
    hyp_lines = by_page.get(page_id)
    if hyp_lines is None:
        continue
    ref_lines = [ln.strip() for ln in gold_text.splitlines() if ln.strip()]
    if not ref_lines:
        continue

    hyp_page_text = _normalize("\n".join(hyp_lines))
    ref_page_text = _normalize("\n".join(ref_lines))
    page_cer = _levenshtein(hyp_page_text, ref_page_text) / len(ref_page_text)
    page_wer = _wer(hyp_page_text.split(), ref_page_text.split()) if ref_page_text.split() else None
    page_results.append({
        'page_id': page_id,
        'ocr_lines': len(hyp_lines),
        'gold_lines': len(ref_lines),
        'page_cer': page_cer,
        'page_wer': page_wer,
        'hyp_page_text': hyp_page_text,
        'ref_page_text': ref_page_text,
    })

    if len(hyp_lines) != len(ref_lines):
        mismatched_pages.append((page_id, len(hyp_lines), len(ref_lines)))
        continue

    for hyp, ref in zip(hyp_lines, ref_lines):
        ref_norm = _normalize(ref)
        if not ref_norm:
            continue
        cer = _levenshtein(hyp, ref_norm) / len(ref_norm)
        hyp_words, ref_words = hyp.split(), ref_norm.split()
        wer = _wer(hyp_words, ref_words) if ref_words else None
        line_results.append({
            'page_id': page_id,
            'cer': cer,
            'wer': wer,
            'hyp': hyp,
            'ref': ref_norm,
        })

print(f'scored {len(line_results)} lines across {len({r["page_id"] for r in line_results})} gold pages')
print(f'scored {len(page_results)} gold pages at page level')
if mismatched_pages:
    print(f'skipped line-level scoring for {len(mismatched_pages)} page(s) with mismatched line counts '
          f'(ocr lines vs. gold lines): {mismatched_pages}')

scored 31 lines across 1 gold pages
scored 5 gold pages at page level
skipped line-level scoring for 4 page(s) with mismatched line counts (ocr lines vs. gold lines): [('bishwaparichay_p0343', 31, 6), ('chandalika_p0161', 26, 27), ('chitrangada_p0130', 26, 31), ('tin_sangi_p0219', 31, 11)]


In [5]:
# 4. Display aggregate CER / WER
import pandas as pd

if line_results:
    mean_cer = sum(r['cer'] for r in line_results) / len(line_results)
    wer_values = [r['wer'] for r in line_results if r['wer'] is not None]
    mean_wer = sum(wer_values) / len(wer_values) if wer_values else float('nan')
    print(f'Line-level mean CER: {mean_cer:.4f}')
    print(f'Line-level mean WER: {mean_wer:.4f}')
    df = pd.DataFrame(line_results)[['page_id', 'cer', 'wer', 'hyp', 'ref']]
    df
else:
    print('No pages had exact line-count matches, so line-level scoring was skipped.')

if page_results:
    mean_page_cer = sum(r['page_cer'] for r in page_results) / len(page_results)
    page_wer_values = [r['page_wer'] for r in page_results if r['page_wer'] is not None]
    mean_page_wer = sum(page_wer_values) / len(page_wer_values) if page_wer_values else float('nan')
    print(f'Page-level mean CER: {mean_page_cer:.4f}')
    print(f'Page-level mean WER: {mean_page_wer:.4f}')
    page_df = pd.DataFrame(page_results)[['page_id', 'ocr_lines', 'gold_lines', 'page_cer', 'page_wer']]
    page_df
else:
    print('Nothing scored -- check that grading_kit/labels.jsonl has real entries and that '
          'the corresponding pages exist under data/raw/.')

Line-level mean CER: 0.0595
Line-level mean WER: 0.2988
Page-level mean CER: 0.0899
Page-level mean WER: 0.3460


In [ ]:
import sys
import tempfile
from pathlib import Path

import yaml

# Make src/ importable -- Section 1 above already does this, but Section 2 is written to also work
# standalone (e.g. if a reader re-runs just from here), so it repeats the same defensive check
# rather than assuming Section 1 already ran in this kernel.
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from doc_agent.contracts import Chunk  # noqa: E402
from doc_agent.index import chunk, embed, store  # noqa: E402

# Load the REAL project config: the MODEL choices below (multilingual-e5-base, FAISS inner-product
# search) are exactly what scripts/build_index.sh would use for the real corpus. Two SCALE-
# dependent settings are overridden just below for this small demo, each with its own reason --
# everything else (embed model/dim, base index metric) is untouched. Named demo_cfg (not cfg) so
# it never collides with Section 1's own `cfg` variable in this shared notebook kernel.
with open(ROOT / "configs" / "config.yaml", encoding="utf-8") as f:
    demo_cfg = yaml.safe_load(f)

demo_dir = Path(tempfile.mkdtemp(prefix="kb_demo_"))
demo_cfg["paths"] = {
    "raw_dir": str(demo_dir / "raw"),
    "processed_dir": str(demo_dir / "processed"),
    "index_dir": str(demo_dir / "processed" / "index"),
}
Path(demo_cfg["paths"]["processed_dir"]).mkdir(parents=True, exist_ok=True)

# config.yaml's production index type is "faiss:hnsw" (M=32 graph links per node) -- correct for
# the real ~1000s-of-chunks corpus, but HNSW needs a graph with enough nodes to be meaningful, and
# this demo produces only a handful of chunks. Confirmed directly (not assumed): faiss-cpu's
# IndexHNSWFlat SEGFAULTS in this environment when .add() is called with fewer vectors than M --
# not a "bug in this notebook," a real rough edge in a graph index built for far more data than a
# demo corpus has. store.py's own test suite (tests/test_retrieval.py) already avoids this the
# same way, using "faiss:flat" for every test. Same fix here, same reason.
demo_cfg["index"]["type"] = "faiss:flat"

# config.yaml's production chunk_tokens/overlap (256/32) are sized for real OCR'd paragraphs -- at
# that size, this demo's 6 short lines (~47 tokens total) would merge into a single chunk, which
# can't demonstrate index statistics across chunks or a retrieval choice among candidates (Sections
# 4-5 below need more than one chunk to be meaningful). Scaled down for THIS demo's size only; the
# real corpus run still uses config.yaml's 256/32 unmodified.
demo_cfg["index"]["chunk_tokens"] = 15
demo_cfg["index"]["overlap"] = 3

print(f"demo scratch dir (gitignored, throwaway): {demo_dir}")
print(f"embed model: {demo_cfg['embed']['model']} (dim={demo_cfg['embed']['dim']})")
print(
    f"index type: {demo_cfg['index']['type']} (demo override, see comment above; "
    f"config.yaml's real value is 'faiss:hnsw')"
)
print(
    f"chunk_tokens={demo_cfg['index']['chunk_tokens']}, overlap={demo_cfg['index']['overlap']} "
    f"(demo override; config.yaml's real values are 256/32)"
)

## 2. The demonstration corpus (synthetic — see the scope note above)

Six hand-written Bengali lines about Rabindranath Tagore himself (thematically appropriate to the
real corpus, but **not** OCR output — written directly as clean Unicode text), across 2 synthetic
pages of one synthetic `doc_id`. Each line gets a made-up `ocr_confidence` and `evidence_tier`
("gold"/"silver"), in the *exact* row shape `vision/ocr.py::transcribe()` writes to
`ocr_meta.jsonl` (`chunk_id`, `ocr_confidence`, `evidence_tier` — see
`reports/pipeline_diagram.md` §2.4 and §4) — this is what makes it possible to feed straight into
the real, unmodified `chunk.split()` below, exactly as if a real OCR pass had produced it.

In [ ]:
import json

doc_id = "rachanabali_vol25_demo"

# (line_suffix, page_suffix, text, ocr_confidence, evidence_tier)
# Confidence/tier values are hand-picked to cover both tiers and a range of confidence, including
# one deliberately low-confidence line (l005) used later by the worst-failure cell.
demo_lines = [
    ("l000", "p0001", "রবীন্দ্রনাথ ঠাকুর ছিলেন একজন বাঙালি কবি ঔপন্যাসিক ও দার্শনিক", 0.94, "gold"),
    ("l001", "p0001", "তিনি ১৯১৩ সালে সাহিত্যে নোবেল পুরস্কার লাভ করেন", 0.91, "gold"),
    ("l002", "p0001", "গীতাঞ্জলি কাব্যগ্রন্থের জন্য তিনি বিশ্বজোড়া খ্যাতি অর্জন করেন", 0.88, "silver"),
    ("l003", "p0002", "শান্তিনিকেতনে তিনি বিশ্বভারতী বিশ্ববিদ্যালয় প্রতিষ্ঠা করেন", 0.85, "silver"),
    ("l004", "p0002", "তাঁর রচনা বাংলা সাহিত্যে গভীর প্রভাব বিস্তার করেছে", 0.79, "silver"),
    ("l005", "p0002", "তিনি ছোটগল্প উপন্যাস নাটক ও প্রবন্ধ রচনা করেছেন", 0.62, "silver"),
]

line_chunks = []
ocr_meta_rows = []
for suffix, page_suffix, text, confidence, tier in demo_lines:
    page_id = f"{doc_id}_{page_suffix}"
    chunk_id = f"{page_id}_{suffix}"
    line_chunks.append(Chunk(id=chunk_id, doc_id=doc_id, text=text, page_ids=[page_id]))
    ocr_meta_rows.append(
        {"chunk_id": chunk_id, "ocr_confidence": confidence, "evidence_tier": tier}
    )

# Written under demo_cfg["paths"]["processed_dir"] -- exactly where the real ocr.transcribe() would
# have written it, so chunk.split() (called unmodified below) can't tell the difference.
ocr_meta_path = Path(demo_cfg["paths"]["processed_dir"]) / "ocr_meta.jsonl"
with open(ocr_meta_path, "w", encoding="utf-8") as f:
    for row in ocr_meta_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

n_pages = len({c.page_ids[0] for c in line_chunks})
print(f"{len(line_chunks)} synthetic line-level chunks across {n_pages} pages, doc_id={doc_id!r}")

## 3. The real Stage 4 chain: chunk → embed → store

From here on, nothing is mocked — this calls the actual `index/chunk.py`, `index/embed.py`, and
`index/store.py` functions, the same ones `scripts/build_index.sh` calls for the real corpus. The
first call to `embed.encode()` downloads `intfloat/multilingual-e5-base` (~1.1GB) from the
Hugging Face Hub if it isn't already cached locally — one-time cost, cached for every future run.

In [ ]:
index_chunks = chunk.split(line_chunks, demo_cfg)
print(f"chunk.split(): {len(line_chunks)} lines -> {len(index_chunks)} merged chunk(s)")
for c in index_chunks:
    print(f"  {c.id}  pages={c.page_ids}  text={c.text[:60]!r}...")

vectors = embed.encode(index_chunks, demo_cfg)
print(f"\nembed.encode(): shape={vectors.shape}, dtype={vectors.dtype}")

store.build(index_chunks, vectors, demo_cfg)
faiss_index, chunk_rows = store.load(demo_cfg)
print(
    f"\nstore.build()/load(): {faiss_index.ntotal} vectors indexed, "
    f"{len(chunk_rows)} chunk records loaded back"
)

## 4. Index statistics (real numbers, demo-scale)

Pulled live from the `faiss_index`/`chunk_rows` built in Section 2 — nothing hardcoded. Labelled
demo-scale throughout: these are real counts for what was actually indexed here, not a stand-in for
the real corpus's coverage (see the scope note in the title cell for why the real run hasn't
happened yet).

In [ ]:
from collections import Counter

n_chunks = faiss_index.ntotal
dim = vectors.shape[1]
index_type = demo_cfg["index"]["type"]
distinct_docs = {row["doc_id"] for row in chunk_rows}
tier_counts = Counter(row["tier"] for row in chunk_rows)
total_pages_covered = {pid for row in chunk_rows for pid in row["page_ids"]}

print(f"chunks indexed:        {n_chunks}")
print(f"embedding dimension:   {dim}")
print(f"index type (demo):     {index_type}  (real corpus run uses config.yaml's 'faiss:hnsw')")
print(f"distinct doc_ids:      {len(distinct_docs)}  ({sorted(distinct_docs)})")
print(f"tier breakdown:        {dict(tier_counts)}")
print(
    f"corpus coverage:       {len(total_pages_covered)} synthetic pages "
    f"(demo-scale — NOT the real 437-page corpus; see scope note in the title cell)"
)

## 5. One real retrieval example (direct FAISS, not `retrieval/retriever.py`)

`retrieval/retriever.py` is A3 scope and still `raise NotImplementedError` by design (see
`reports/pipeline_diagram.md` — its `top_score()`/`is_weak()`/`next_k()` helpers are filled in as
A3 plumbing, but `Retriever.retrieve()` itself is not built yet). This cell demonstrates retrieval
directly against the index built above instead: `store.load()` + a query embedding +
`faiss_index.search()`.

One deliberate detail: `embed.encode()` always applies e5's `"passage: "` prefix (it's written for
indexing text, not querying it) — e5's asymmetric scheme expects `"query: "` on the query side
instead. `retriever.py` will own that logic for real in A3; here, for this one illustrative query,
the prefix is applied manually via the same cached model `embed._load_model()` returns, rather than
duplicating retriever.py's future responsibility.

In [ ]:
# A real query, semantically close to (but not copied verbatim from) chunk c00002's text about
# Tagore founding Visva-Bharati University at Shantiniketan.
query_text = "শান্তিনিকেতনে বিশ্ববিদ্যালয় প্রতিষ্ঠা"

model = embed._load_model(demo_cfg["embed"]["model"], embed._resolve_device(demo_cfg))
query_vector = model.encode([f"query: {query_text}"], normalize_embeddings=True).astype("float32")

k = min(3, faiss_index.ntotal)
scores, ids = faiss_index.search(query_vector, k)

print(f"query: {query_text!r}\n")
for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), start=1):
    row = chunk_rows[idx]
    print(f"#{rank}  score={score:.4f}  id={row['id']}  pages={row['page_ids']}")
    print(f"      text: {row['text']}\n")

top = chunk_rows[ids[0][0]]
right_page = "rachanabali_vol25_demo_p0002" in top["page_ids"]
print(f"Top hit on the right page (p0002, where the university-founding line lives)? {right_page}")

## 6. Worst failure — retrieval-level (index/chunk/embed/store's own evidence)

A concrete retrieval failure demonstrated on the demo index built above — the OCR-level worst
failure is a separate placeholder for Person B, see the final section below.

In [ ]:
# A short, generic, pronoun-heavy query that closely echoes chunk c00000's own opening phrasing
# ("...ছিলেন একজন...কবি..." / "...was a...poet...") -- the obviously-intended match. Verified
# directly (not assumed): it does NOT come back top-1.
query_text = "তিনি একজন কবি ছিলেন"  # "He was a poet"

query_vector = model.encode([f"query: {query_text}"], normalize_embeddings=True).astype("float32")
scores, ids = faiss_index.search(query_vector, faiss_index.ntotal)

print(f"query: {query_text!r}  (closely echoes c00000's own text)\n")
for rank, (score, idx) in enumerate(zip(scores[0], ids[0]), start=1):
    row = chunk_rows[idx]
    flag = "  <-- expected top-1, but isn't" if row["id"].endswith("c00000") else ""
    print(f"#{rank}  score={score:.4f}  id={row['id']}{flag}")
    print(f"      text: {row['text']}\n")

**Read on why:** the query is short and pronoun-heavy (`তিনি` = "he", generic), and every chunk in
this tiny 4-chunk demo corpus contains that same pronoun somewhere (all four are about Tagore). At
this scale, a 3-4 word query doesn't carry enough distinguishing signal for `multilingual-e5-base`
to separate "the chunk that literally opens with this phrase" from "any other chunk about the same
person" — cosine similarity across all four candidates sits within a tight band (see the scores
above; expect small run-to-run float differences in the exact ranking, but the "obvious" match
consistently lands behind at least one less-obvious candidate, never a clean #1). This is a real
limitation consistent with retrieval quality on short/generic queries generally (not specific to
Bengali or to this demo), and it's exactly the kind of case `configs/design_choices.md`'s Stage 5
note and the A2 form's Section 6 flag as the biggest open risk for A3: whether
`multilingual-e5-base` retrieval holds up well enough on short or ambiguous real queries once
reranking (`cfg.retrieve.rerank`) and evidence-gated re-search (widen `k` on weak top-score, per
`retriever.py`'s already-stubbed `is_weak()`/`next_k()`) are added on top in A3 — those exist
specifically to catch cases like this one.

## 7. Placeholder for Person B (OCR-level worst failure)

**Intentionally left unfilled below.** Section 1 above now covers real OCR quality (CER/WER
against real gold labels) -- that part of the original placeholder is done. The one remaining
item is an **OCR-level worst-failure example**: one real line where OCR reading failed badly, with
a read on why. `configs/design_choices.md`'s Stage 3 methods note flags Bengali conjunct-consonant
(যুক্তাক্ষর) and matra segmentation as the likeliest risk area to check first -- Section 1's own
`line_results`/`page_results` (sorted by `cer` descending) are the natural place to pull a concrete
example from.

In [ ]:
# TODO(Person B): pick the worst-scoring real line from Section 1's line_results (sort by
# 'cer' descending) and explain why it failed -- see the markdown cell above.
# Left as a loud, explicit stub (matching this repo's own `# IMPLEMENT` / NotImplementedError
# convention) rather than silently empty, so it can't be mistaken for "done" if run as-is.
raise NotImplementedError(
    "Person B: OCR-level worst-failure example -- see markdown cell above"
)